# CS781 Phase 2 - Legal NER on top of the Phase-1 models

One notebook for every Phase-2 experiment. Set the experiment in **section 3** and run all, once per experiment.

| switch | options |
|---|---|
| `MODEL_TYPE` | `"gpt"`, `"modern"`, `"rnn"`, `"t5"` (your Phase-1 models; for `t5` only the encoder is used) |
| `FT_METHOD` | `"full"`, `"lora"`, `"lorma"`, `"partial"` (top layers only), `"head_only"` (linear probe) |
| `USE_KD` | `True`: train a smaller student (first `STUDENT_LAYERS` layers of `MODEL_TYPE`) from a saved teacher |

Pipeline: text -> words -> pieces -> BPE tokens **with character offsets** -> BIO tag per token -> sliding windows -> Phase-1 model + linear tag head -> tags -> spans -> character offsets -> `prediction.json`.
Section 7 checks the offset code: turning the gold spans into tags and back must give the gold spans again.

## 1. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("hf_token")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
  print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Configuration

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import gc
import bisect

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_TEXT_FILES = ["Phase_2_LNER/Train/fold1-text.json", "Phase_2_LNER/Train/fold2-text.json"]
TRAIN_GOLD_FILES = ["Phase_2_LNER/Train/fold1-gold.json", "Phase_2_LNER/Train/fold2-gold.json"]
LABELS_FILE = "Phase_2_LNER/Train/labels.txt"
TEST_TEXT_FILE = "Phase_2_LNER/Test/fold3-text.json"

# ---------------- the experiment ----------------
EXPERIMENT_NAME = "modern_dec_nomerge"
MODEL_TYPE = "modern"        # gpt | modern | rnn | t5
FT_METHOD = "full"        # full | lora | lorma | partial | head_only
SMOKE_TEST = False

# knowledge distillation (Section 8.2.2): student = first STUDENT_LAYERS layers of MODEL_TYPE
USE_KD = False
STUDENT_LAYERS = 2
TEACHER_MODEL_TYPE = "modern"
TEACHER_FT_METHOD = "full"
TEACHER_NER_PATH = "/kaggle/input/<teacher-notebook>/cs781_phase2/modern_full/ner_model.pt"
KD_TEMPERATURE = 2.0
# (the teacher must have been trained with the same WINDOW_TOKENS, SPLIT_MODE and HEAD_LOOKAHEAD as this run)
KD_ALPHA = 0.5               # loss = KD_ALPHA * CE(gold tags) + (1 - KD_ALPHA) * T^2 * KL(teacher || student)

# LoRA / LoRMA
ADAPTER_RANK = 8
ADAPTER_ALPHA = 16
PARTIAL_TOP_LAYERS = 2       # "partial": only the top layers (+ final norm + head) are trained

# Phase-1 models: one Kaggle dataset with model_<type>.safetensors and bpe_legal_<type>.json
PHASE1_FOLDER = "/kaggle/input/datasets/medinim/phase1-models"        # copy the exact path from the right-hand panel
PHASE1_MODEL_FILES = {k: os.path.join(PHASE1_FOLDER, f"model_{k}.safetensors") for k in ["gpt", "modern", "rnn", "t5"]}
PHASE1_BPE_FILES = {k: os.path.join(PHASE1_FOLDER, f"bpe_legal_{k}.json") for k in ["gpt", "modern", "rnn", "t5"]}

# Phase-1 architecture sizes: must be EXACTLY what Phase 1 trained with
MODEL_CONTEXT_LENGTH = 2048
N_EMBD, N_LAYER, N_HEAD, N_KV_HEAD = 640, 10, 10, 5
D_FF = 4 * N_EMBD
RNN_CELL, RNN_HIDDEN, RNN_LAYERS = "lstm", 1152, 4
T5_D_MODEL, T5_ENC_LAYERS, T5_DEC_LAYERS, T5_HEADS, T5_D_FF, T5_N_SENTINELS = 512, 8, 8, 8, 2048, 100
DROPOUT = 0.1

# data / context
WINDOW_TOKENS = 512          # context window fed to the model
STRIDE = 384                 # windows overlap by WINDOW_TOKENS - STRIDE tokens
SPLIT_MODE = "edges"         # "none" | "edges" (split punctuation off word edges) | "all" (every punctuation mark separate)
VAL_FRACTION = 0.15          # documents held out from fold1+fold2 for local evaluation
O_WEIGHT = 1.0               # class balancing: loss weight of the "O" tag (e.g. 0.5 to push recall)
HEAD_LOOKAHEAD = 3           # prediction head: 0 = linear head on h_t; K > 0 = linear head on [h_t ; h_t+1 ; ... ; h_t+K]
                             # (lets a left-to-right model see K tokens to the right before tagging token t)
DECODE_MERGE = True     # best run: merge decoding on
SNAP_TO_WORDS = True         # span decoding: spans never start/end inside a run of letters/digits
LOAD_NER_PATH = None    # None = train modern full FT; or a saved ner_model.pt to only re-decode

# training (Trainer, like Phase 1)
if FT_METHOD in ("full", "partial") or USE_KD:
  LEARNING_RATE = 1e-4
else:
  LEARNING_RATE = 1e-3       # adapters and a fresh head can take a bigger learning rate
EPOCHS = 15
BATCH_SIZE = 8
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if SMOKE_TEST:
  EPOCHS = 1

OUTPUT_DIR = "/kaggle/working/cs781_phase2"
EXP_DIR = os.path.join(OUTPUT_DIR, EXPERIMENT_NAME)
os.makedirs(EXP_DIR, exist_ok=True)
PREDICTION_PATH = os.path.join(EXP_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, f"submission_{EXPERIMENT_NAME}.zip")
RESULTS_PATH = os.path.join(OUTPUT_DIR, "results.jsonl")
CAUSAL = MODEL_TYPE in ("gpt", "modern", "rnn")   # left-to-right models only see the past
print("Experiment:", EXPERIMENT_NAME, "|", MODEL_TYPE, FT_METHOD, "| KD" if USE_KD else "")

## 4. Tokenizer (the Phase-1 BPE, unchanged)

In [ ]:
import json
import heapq
from collections import Counter, defaultdict

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
WORD_END = "</w>"

class ByteLevelBPE:
  def __init__(self, vocab_size):
      self.vocab_size = vocab_size
      self.merges = []
      self.merge_ranks = {}
      self.itos = []
      self.stoi = {}
      self._cache = {}
      self.pad_id = None
      self.bos_id = None
      self.eos_id = None

  def _word_to_symbols(self, word):
      byte_vals = word.encode("utf-8")
      symbols = [chr(b) for b in byte_vals]
      symbols.append(WORD_END)
      return symbols

  def train(self, sentences):
      word_freq = Counter()
      for sent in sentences:
          for word in sent.split():
              word_freq[word] += 1

      words = list(word_freq)
      freqs = [word_freq[w] for w in words]
      word_symbols = [self._word_to_symbols(w) for w in words]

      vocab_list = list(SPECIALS)
      vocab_seen = set(vocab_list)

      initial_symbols = {chr(b) for b in range(256)} | {WORD_END}

      for s in sorted(initial_symbols):
          if s not in vocab_seen:
              vocab_list.append(s)
              vocab_seen.add(s)

      # this loop recounts every pair every merge, which is far too slow
      # for the legal corpus. Replace it with incremental pair counts + pair_to_words.
      pair_counts = Counter()
      pair_to_words = defaultdict(set)
      for wi, symbols in enumerate(word_symbols):
          f = freqs[wi]
          for i in range(len(symbols) - 1):
              p = (symbols[i], symbols[i + 1])
              pair_counts[p] += f
              pair_to_words[p].add(wi)

      heap = [(-c, p) for p, c in pair_counts.items()]
      heapq.heapify(heap)

      while len(vocab_list) < self.vocab_size:
          best_pair, best_count = None, 0
          while heap:
              negc, p = heapq.heappop(heap)
              if pair_counts.get(p, 0) == -negc and -negc > 0:
                  best_pair, best_count = p, -negc
                  break
          if best_pair is None or best_count <= 1:
              break

          a, b = best_pair
          merged = a + b
          changed = set()

          for wi in list(pair_to_words[best_pair]):
              symbols = word_symbols[wi]
              f = freqs[wi]

              old_pairs = Counter(zip(symbols, symbols[1:]))   # positions, so overlaps count
              new_symbols = []
              i = 0
              while i < len(symbols):
                  if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                      new_symbols.append(merged)
                      i += 2
                  else:
                      new_symbols.append(symbols[i])
                      i += 1
              word_symbols[wi] = new_symbols
              new_pairs = Counter(zip(new_symbols, new_symbols[1:]))

              for p, c in old_pairs.items():
                  pair_counts[p] -= c * f
                  changed.add(p)
                  if p not in new_pairs:
                      pair_to_words[p].discard(wi)
              for p, c in new_pairs.items():
                  pair_counts[p] += c * f
                  changed.add(p)
                  pair_to_words[p].add(wi)

          for p in changed:
              c = pair_counts[p]
              if c > 0:
                  heapq.heappush(heap, (-c, p))
              else:
                  del pair_counts[p]
                  pair_to_words.pop(p, None)

          self.merges.append([a, b])
          self.merge_ranks[(a, b)] = len(self.merges) - 1

          if merged not in vocab_seen:
              vocab_list.append(merged)
              vocab_seen.add(merged)

      self.itos = vocab_list
      self.stoi = {tok: i for i, tok in enumerate(self.itos)}
      self.pad_id = self.stoi["<pad>"]
      self.bos_id = self.stoi["<s>"]
      self.eos_id = self.stoi["</s>"]
      return self

  def encode_word(self, word):
      if word in self._cache:
        return self._cache[word]
      symbols = self._word_to_symbols(word)
      while len(symbols) > 1:
          pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
          ranked = [(self.merge_ranks[p], i) for i, p in enumerate(pairs) if p in self.merge_ranks]
          if not ranked:
              break
          _, i = min(ranked)
          symbols = symbols[:i] + [symbols[i] + symbols[i + 1]] + symbols[i + 2:]

      self._cache[word] = symbols
      return symbols

  def encode_ids(self, sentence):
      unk = self.stoi["<unk>"]
      return [self.stoi.get(t, unk) for t in self.encode(sentence)]

  def decode(self, ids):
      text = "".join(self.itos[i] for i in ids if self.itos[i] not in SPECIALS)
      words = text.split(WORD_END)
      out = [bytes(ord(c) for c in w).decode("utf-8", errors="replace") for w in words if w]
      return " ".join(out)

  def encode(self, sentence):
      tokens = []
      for word in sentence.split():
          tokens.extend(self.encode_word(word))
      return tokens

  def save(self, path):
      with open(path, "w", encoding="utf-8") as f:
          json.dump({"merges": self.merges, "itos": self.itos,
                      "vocab_size": self.vocab_size}, f, ensure_ascii=False)

  @staticmethod
  def load_bpe(path):
      with open(path, "r", encoding="utf-8") as f:
          data = json.load(f)
      bpe = ByteLevelBPE(vocab_size=data["vocab_size"])
      bpe.merges = data["merges"]
      bpe.itos = data["itos"]
      bpe.stoi = {tok: i for i, tok in enumerate(bpe.itos)}
      bpe.merge_ranks = {(a, b): i for i, (a, b) in enumerate(bpe.merges)}
      bpe.pad_id = bpe.stoi["<pad>"]
      bpe.bos_id = bpe.stoi["<s>"]
      bpe.eos_id = bpe.stoi["</s>"]
      return bpe

In [ ]:
import hashlib
for k, f in PHASE1_BPE_FILES.items():   # all four must be the same tokenizer (needed for KD)
  print(k, hashlib.md5(open(f, "rb").read()).hexdigest())
bpe = ByteLevelBPE.load_bpe(PHASE1_BPE_FILES[MODEL_TYPE])
UNK_ID = bpe.stoi["<unk>"]
print("vocab size:", len(bpe.itos))

## 5. Download the Phase-2 data

In [ ]:
from huggingface_hub import hf_hub_download

def load_json_file(fname):
  p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=fname, repo_type="dataset", token=HF_TOKEN)
  with open(p, "r", encoding="utf-8") as f:
    return json.load(f)

def as_text(v):
  return v["text"] if isinstance(v, dict) else v

texts = {}
gold_raw = {}
for tf, gf in zip(TRAIN_TEXT_FILES, TRAIN_GOLD_FILES):
  for k, v in load_json_file(tf).items():
    texts[str(k)] = as_text(v)
  for k, v in load_json_file(gf).items():
    gold_raw[str(k)] = v
test_texts = {str(k): as_text(v) for k, v in load_json_file(TEST_TEXT_FILE).items()}

labels_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=LABELS_FILE, repo_type="dataset", token=HF_TOKEN)
LABELS = [l.strip() for l in open(labels_path, encoding="utf-8") if l.strip() and l.strip() != "O"]

print("train documents:", len(texts), "| test documents:", len(test_texts))
print("labels:", LABELS)
first = list(gold_raw.keys())[0]
print("example annotation format:", json.dumps(gold_raw[first])[:300])

## 6. Read the annotations and check the offset convention

The brief does not say whether `end` is exclusive (Python style, `text[start:end]`) or inclusive. We check it against the entity text: we use exclusive ends internally, and write predictions back in the convention the gold files use.

In [ ]:
def read_annotation(a):
  if isinstance(a, dict):
    s = a.get("start", a.get("start_offset", a.get("begin")))
    e = a.get("end", a.get("end_offset"))
    lab = a.get("label", a.get("type", a.get("tag")))
    txt = a.get("text", a.get("entity_text"))
  else:
    s, e, lab = a[0], a[1], a[2]
    txt = a[3] if len(a) > 3 else None
  return int(s), int(e), str(lab), txt

def annotation_list(v):
  if isinstance(v, dict):
    v = v.get("entities", v.get("annotations", v.get("spans", [])))
  return [read_annotation(a) for a in v]

gold_read = {doc: annotation_list(v) for doc, v in gold_raw.items() if doc in texts}

n_excl, n_incl, n_other = 0, 0, 0
bad_examples = []
for doc, anns in gold_read.items():
  t = texts[doc]
  for s, e, lab, txt in anns:
    if txt is None:
      continue
    if t[s:e] == txt:
      n_excl += 1
    elif t[s:e + 1] == txt:
      n_incl += 1
    else:
      n_other += 1
      if len(bad_examples) < 5:
        bad_examples.append((doc, s, e, lab, txt, t[s:e + 1]))
END_INCLUSIVE = n_incl > n_excl
print(f"text[s:e] == entity text: {n_excl} | text[s:e+1] == entity text: {n_incl} | neither: {n_other}")
print("END_INCLUSIVE =", END_INCLUSIVE)
for b in bad_examples:
  print("  mismatch:", b)

# internal format: (start, end_exclusive, label)
gold = {}
unknown_labels = {}
for doc, anns in gold_read.items():
  spans = []
  for s, e, lab, txt in anns:
    if lab not in LABELS:
      unknown_labels[lab] = unknown_labels.get(lab, 0) + 1
      continue
    spans.append((s, e + 1 if END_INCLUSIVE else e, lab))
  gold[doc] = sorted(set(spans))
if unknown_labels:
  print("annotations with labels not in labels.txt (skipped):", unknown_labels)

label_counts = {l: 0 for l in LABELS}
for spans in gold.values():
  for s, e, lab in spans:
    label_counts[lab] += 1
print("entities per label (class imbalance):", label_counts)
lengths = sorted(len(texts[d]) for d in gold)
print(f"document length in characters: min {lengths[0]}, median {lengths[len(lengths)//2]}, max {lengths[-1]}")

# validation split by document
doc_ids = sorted(gold.keys())
rng = random.Random(SEED)
rng.shuffle(doc_ids)
n_val = max(1, int(len(doc_ids) * VAL_FRACTION))
val_docs = sorted(doc_ids[:n_val])
train_docs = sorted(doc_ids[n_val:])
print("train docs:", len(train_docs), "| val docs:", len(val_docs))

## 7. Character-offset tokenization, BIO tags, and the offset test

* **Words:** split exactly like `str.split()` (your BPE splits text the same way), but we keep each word's start and end character.
* **Pieces:** `SPLIT_MODE = "edges"` cuts punctuation off the start and end of a word (`(Ram,` becomes `(`, `Ram`, `,`), so an entity can end before a comma. `"all"` makes every punctuation mark its own piece.
* **Tokens:** each piece goes through `bpe.encode_word`. Every BPE symbol is a run of UTF-8 bytes, which we map back to characters, so every token gets exact `[start, end)` character offsets. The bare `</w>` symbol covers no characters: it gets the label -100 (ignored) and is skipped when we decode.
* **Tags:** `B-LABEL` on the first token of an entity, `I-LABEL` on the rest, `O` everywhere else.

In [ ]:
TAGS = ["O"] + [p + "-" + l for l in LABELS for p in ("B", "I")]
TAG2ID = {t: i for i, t in enumerate(TAGS)}
PUNCT = set(".,;:()[]{}\"'`-/\u201c\u201d\u2018\u2019")

def split_words_with_offsets(text):
  # the same rule as str.split(): runs of non-whitespace characters
  words = []
  i, n = 0, len(text)
  while i < n:
    while i < n and text[i].isspace():
      i += 1
    if i >= n:
      break
    j = i
    while j < n and not text[j].isspace():
      j += 1
    words.append((i, j))
    i = j
  return words

def split_pieces(text, s, e):
  if SPLIT_MODE == "none":
    return [(s, e)]
  if SPLIT_MODE == "all":
    pieces, start = [], s
    for i in range(s, e):
      if text[i] in PUNCT:
        if start < i:
          pieces.append((start, i))
        pieces.append((i, i + 1))
        start = i + 1
    if start < e:
      pieces.append((start, e))
    return pieces
  # "edges"
  front, back = [], []
  while e - s > 1 and text[s] in PUNCT:
    front.append((s, s + 1))
    s += 1
  while e - s > 1 and text[e - 1] in PUNCT:
    back.insert(0, (e - 1, e))
    e -= 1
  return front + [(s, e)] + back

def tokenize_with_offsets(text):
  ids, starts, ends = [], [], []
  for ws, we in split_words_with_offsets(text):
    for s, e in split_pieces(text, ws, we):
      piece = text[s:e]
      byte_to_char = []
      for ci, ch in enumerate(piece):
        byte_to_char.extend([ci] * len(ch.encode("utf-8")))
      byte_to_char.append(len(piece))
      pos = 0
      for sym in bpe.encode_word(piece):
        nb = len(sym) - len(WORD_END) if sym.endswith(WORD_END) else len(sym)   # bytes covered by this token
        ids.append(bpe.stoi.get(sym, UNK_ID))
        if nb > 0:
          starts.append(s + byte_to_char[pos])
          ends.append(s + byte_to_char[pos + nb])
        else:
          starts.append(e)        # bare </w>: covers no characters
          ends.append(e)
        pos += nb
  return ids, starts, ends

def make_tags(starts, ends, spans):
  # -100 = ignored by the loss (tokens that cover no characters)
  tags = [0 if ends[t] > starts[t] else -100 for t in range(len(starts))]
  n_conflict = 0
  for s, e, lab in sorted(spans, key=lambda x: (x[0], -x[1])):
    first = True
    t = bisect.bisect_left(starts, s)
    while t < len(starts) and starts[t] < e:
      if ends[t] > starts[t]:
        if tags[t] == 0:
          tags[t] = TAG2ID[("B-" if first else "I-") + lab]
        else:
          n_conflict += 1           # overlapping / nested entities cannot both be BIO-tagged
        first = False
      t += 1
  return tags, n_conflict

def decode_spans(tag_ids, starts, ends):
  spans, cur = [], None             # cur = [start, end, {label: count}]
  for t, tid in enumerate(tag_ids):
    if ends[t] <= starts[t]:
      continue                      # empty tokens never start, end or break a span
    tag = TAGS[tid]
    if tag == "O":
      if cur:
        spans.append(cur)
      cur = None
      continue
    kind, lab = tag.split("-", 1)
    continues = cur is not None and kind == "I" and (DECODE_MERGE or lab in cur[2])
    if continues:
      cur[1] = ends[t]
      cur[2][lab] = cur[2].get(lab, 0) + 1
    else:                           # B-, or an I- that cannot continue the current span
      if cur:
        spans.append(cur)
      cur = [starts[t], ends[t], {lab: 1}]
  if cur:
    spans.append(cur)
  out = []
  for s, e, votes in spans:
    lab = max(votes, key=votes.get)    # majority label of the span's tokens
    out.append((s, e, lab))
  return out

def same_kind(a, b):
  return (a.isalpha() and b.isalpha()) or (a.isdigit() and b.isdigit())

def snap_spans(text, spans):
  # a span may not start or end in the middle of a run of letters (or a run of digits)
  if not SNAP_TO_WORDS:
    return spans
  out = set()
  for s, e, lab in spans:
    while s > 0 and same_kind(text[s - 1], text[s]):
      s -= 1
    while e < len(text) and same_kind(text[e - 1], text[e]):
      e += 1
    out.add((s, e, lab))
  return sorted(out)

def strict_scores(gold_by_doc, pred_by_doc):
  counts = {l: [0, 0, 0] for l in LABELS}          # tp, fp, fn
  for doc in gold_by_doc:
    g = set(gold_by_doc[doc])
    p = set(pred_by_doc.get(doc, []))
    for l in LABELS:
      gl = {x for x in g if x[2] == l}
      pl = {x for x in p if x[2] == l}
      counts[l][0] += len(gl & pl)
      counts[l][1] += len(pl - gl)
      counts[l][2] += len(gl - pl)
  per_label = {}
  for l, (tp, fp, fn) in counts.items():
    if tp + fp + fn == 0:
      continue                                     # label never appears in gold or predictions
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    per_label[l] = (p, r, f, tp + fn)
  macro_p = sum(v[0] for v in per_label.values()) / len(per_label)
  macro_r = sum(v[1] for v in per_label.values()) / len(per_label)
  macro_f = sum(v[2] for v in per_label.values()) / len(per_label)
  return macro_p, macro_r, macro_f, per_label

# tokenize every document once
t0 = time.time()
doc_tok = {}
for doc in list(texts.keys()) + list(test_texts.keys()):
  t = texts.get(doc, test_texts.get(doc))
  doc_tok[doc] = tokenize_with_offsets(t)
print(f"tokenized {len(doc_tok)} documents in {time.time() - t0:.0f}s |",
      "tokens per document (median):", sorted(len(v[0]) for v in doc_tok.values())[len(doc_tok) // 2])

# ---- OFFSET TEST: gold spans -> tags -> spans must give the gold spans back ----
roundtrip = {}
total_conflicts = 0
for doc in gold:
  ids, starts, ends = doc_tok[doc]
  tags, nc = make_tags(starts, ends, gold[doc])
  total_conflicts += nc
  roundtrip[doc] = snap_spans(texts[doc], decode_spans([max(x, 0) for x in tags], starts, ends))
    
rp, rr, rf, rper = strict_scores(gold, roundtrip)
print(f"\nOFFSET TEST (SPLIT_MODE={SPLIT_MODE!r}): best possible strict macro-F1 with this tokenization = {rf:.4f}")
print("tokens with overlapping entities:", total_conflicts)
for l, (p, r, f, n) in rper.items():
  print(f"  {l:10s} reachable recall {r:.3f}  ({n} gold spans)")
shown = 0
for doc in gold:
  missed = set(gold[doc]) - set(roundtrip[doc])
  for s, e, lab in sorted(missed)[:2]:
    if shown < 8:
      print("  cannot represent:", lab, repr(texts[doc][max(0, s - 10):e + 10]), "-> gold:", repr(texts[doc][s:e]))
      shown += 1

## 8. Training windows and datasets

Documents are much longer than the context, so we cut every document into windows of `WINDOW_TOKENS` tokens that overlap. At prediction time each token takes its tag from the window where it has the most context. For the left-to-right models (gpt, modern, rnn) that is the window where the token sits furthest to the right; for the T5 encoder it is the window where the token is closest to the centre.

In [ ]:
from datasets import Dataset

def window_starts(n):
  starts, st = [], 0
  while True:
    starts.append(st)
    if st + WINDOW_TOKENS >= n:
      break
    st += STRIDE
  return starts

def make_examples(docs):
  input_ids, labels = [], []
  for doc in docs:
    ids, starts, ends = doc_tok[doc]
    tags, _ = make_tags(starts, ends, gold[doc])
    for st in window_starts(len(ids)):
      x = ids[st:st + WINDOW_TOKENS]
      y = tags[st:st + WINDOW_TOKENS]
      pad = WINDOW_TOKENS - len(x)
      input_ids.append(x + [bpe.pad_id] * pad)          # right padding
      labels.append(y + [-100] * pad)
  return input_ids, labels

tr_x, tr_y = make_examples(train_docs)
va_x, va_y = make_examples(val_docs)
if SMOKE_TEST:
  tr_x, tr_y = tr_x[:64], tr_y[:64]
train_dataset = Dataset.from_dict({"input_ids": tr_x, "labels": tr_y})
val_dataset = Dataset.from_dict({"input_ids": va_x, "labels": va_y})
print(train_dataset)
print(val_dataset)

def data_collator(features):
  return {"input_ids": torch.tensor([f["input_ids"] for f in features], dtype=torch.long),
          "labels": torch.tensor([f["labels"] for f in features], dtype=torch.long)}

## 9. Phase-1 model code (copied unchanged from the Phase-1 notebook)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    """Module 2.3:  RMS(x) = sqrt( (1/E) * sum_i x_i^2 + eps ),   RMSNorm(x)_i = gamma_i * x_i / RMS(x)"""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))   # learnable, no bias (beta)

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.gamma * (x / rms)


# ---- RoPE (Module 2.3 / 3.1, Assignment 3 "rope.py") ----
def build_rope_cache(max_seq_len, d_head, base=10000.0):
    # theta_i = base^(-2(i-1)/d) for every pair of dimensions; angle = position m * theta_i
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))   # (d_head/2,)
    positions = torch.arange(max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)                                  # (T, d_head/2)
    angles = torch.cat([freqs, freqs], dim=-1)                                # (T, d_head)
    return angles.cos(), angles.sin()


def rotate_half(x):
    # split the last dim in half and return (-x2, x1)
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    # x: (B, n_heads, T, d_head); cos, sin: (B, 1, T, d_head)
    return x * cos + rotate_half(x) * sin


class CausalSelfAttention(nn.Module):
    """Masked multi-head self-attention (Module 2.3, Assignment 3).
    n_kv_heads == n_heads -> normal MHA,  n_kv_heads < n_heads -> grouped-query attention (GQA).
    past_kv = (k, v) of earlier tokens -> KV cache for generation (Module 2.3).
    attention_mask (B, S): 1 = real token, 0 = padding (Module 2.3: masks prevent attention over padding)."""
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.1, use_rope=False):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        self.attention_scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)
        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, position_ids, attention_mask=None, past_kv=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            cos = self.rope_cos[position_ids].unsqueeze(1)   # (B, 1, T, d_head)
            sin = self.rope_sin[position_ids].unsqueeze(1)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past_kv is not None:                  # KV cache: reuse keys/values of earlier tokens
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        new_kv = (k, v)

        if self.n_rep > 1:                       # GQA: each k/v head is shared by n_rep query heads
            k = torch.repeat_interleave(k, self.n_rep, dim=1)
            v = torch.repeat_interleave(v, self.n_rep, dim=1)

        S = k.size(2)                             # number of keys = cached + new
        scores = (q @ k.transpose(-2, -1)) * self.attention_scale        # (B, H, T, S)
        allowed = self.causal_mask[S - T:S, :S].unsqueeze(0).unsqueeze(0)  # (1, 1, T, S)
        if attention_mask is not None:
            allowed = allowed & attention_mask[:, None, None, :].bool()
            # a left-padding position has no real token before it, so let every
            # position always see itself (otherwise its whole row would be masked)
            itself = torch.eye(S, dtype=torch.bool, device=x.device)[S - T:S, :S]
            allowed = allowed | itself.unsqueeze(0).unsqueeze(0)
        scores = scores.masked_fill(~allowed, float("-inf"))
        att = F.softmax(scores, dim=-1)
        att = self.attn_dropout(att)
        out = att @ v                                                      # (B, H, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))
        return out, new_kv


class FeedForward(nn.Module):
    """Assignment 3: two linear layers with a GELU in between, plus dropout."""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class SwiGLUFeedForward(nn.Module):
    """Module 3.1 names SwiGLU as the modern feed-forward activation:
    FFN(x) = W2( SiLU(W1 x) * (W3 x) ),  SiLU(a) = a * sigmoid(a)"""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w3 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        a = self.w1(x)
        return self.dropout(self.w2(a * torch.sigmoid(a) * self.w3(x)))

In [ ]:
from dataclasses import dataclass


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 640
    n_layers: int = 10
    n_heads: int = 10
    n_kv_heads: int = 10            # == n_heads for plain MHA
    d_ff: int = 2560
    max_seq_len: int = 2048
    dropout: float = 0.1
    pos_emb_type: str = "learned"   # "learned" (GPT-2 / BERT, Module 2.3) or "rope"
    norm_type: str = "layernorm"    # "layernorm" or "rmsnorm"
    ffn_type: str = "gelu"          # "gelu" or "swiglu"


class Block(nn.Module):
    """Pre-norm block (Module 2.3, used by GPT-2 and modern LLMs):  x + Attn(Norm(x)),  then  x + FFN(Norm(x))"""
    def __init__(self, cfg):
        super().__init__()
        if cfg.norm_type == "layernorm":
            self.ln1 = nn.LayerNorm(cfg.d_model)
            self.ln2 = nn.LayerNorm(cfg.d_model)
        else:
            self.ln1 = RMSNorm(cfg.d_model)
            self.ln2 = RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"))
        if cfg.ffn_type == "gelu":
            self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        else:
            self.ffn = SwiGLUFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, position_ids, attention_mask=None, past_kv=None):
        a, new_kv = self.attn(self.ln1(x), position_ids, attention_mask, past_kv)
        x = x + a
        x = x + self.ffn(self.ln2(x))
        return x, new_kv


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "learned":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)    # learned absolute positions
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        if cfg.norm_type == "layernorm":
            self.ln_f = nn.LayerNorm(cfg.d_model)
        else:
            self.ln_f = RMSNorm(cfg.d_model)
        # Module 2.1 "Proper Weight Initialization": Xavier/Glorot for the embedding matrices.
        # PyTorch's default N(0, 1) embedding + weight tying gives huge first logits (start loss ~80)
        nn.init.xavier_uniform_(self.tok_emb.weight)
        if cfg.pos_emb_type == "learned":
            nn.init.xavier_uniform_(self.pos_emb.weight)

    def forward(self, input_ids, labels=None, attention_mask=None, position_ids=None,
                past_key_values=None, use_cache=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        if position_ids is None:
            position_ids = torch.arange(past_len, past_len + T, device=input_ids.device).unsqueeze(0).expand(B, T)

        x = self.tok_emb(input_ids)
        if self.cfg.pos_emb_type == "learned":
            x = x + self.pos_emb(position_ids)
        x = self.drop(x)

        new_past = []
        for i, block in enumerate(self.blocks):
            layer_past = None if past_key_values is None else past_key_values[i]
            x, kv = block(x, position_ids, attention_mask, layer_past)
            new_past.append(kv)
        x = self.ln_f(x)

        if use_cache:
            x = x[:, -1:, :]              # generation only needs the prediction for the last position
        logits = x @ self.tok_emb.weight.t()   # weight tying

        loss = None
        if labels is not None:
            # predict token t+1 from positions <= t (labels are shifted inside, like GPT-2)
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)).float(),
                                   labels[:, 1:].reshape(-1))
        if use_cache:
            return {"loss": loss, "logits": logits, "past_key_values": new_past}
        return {"loss": loss, "logits": logits}

In [ ]:
class LSTMCell(nn.Module):
    """One time step of an LSTM (Assignment 2): packed linear layers compute f, i, g, o at once."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x2g = nn.Linear(input_size, 4 * hidden_size)
        self.h2g = nn.Linear(hidden_size, 4 * hidden_size)
        with torch.no_grad():
            self.x2g.bias[0:hidden_size] = 1.0   # forget gate bias close to 1 (Module 2.1)
            self.h2g.bias[0:hidden_size] = 0.0

    def forward(self, x_t, state):
        h_prev, c_prev = state
        gates = self.x2g(x_t) + self.h2g(h_prev)
        f_pre, i_pre, g_pre, o_pre = gates.chunk(4, dim=-1)
        f_t = torch.sigmoid(f_pre)
        i_t = torch.sigmoid(i_pre)
        g_t = torch.tanh(g_pre)
        o_t = torch.sigmoid(o_pre)
        c_t = f_t * c_prev + i_t * g_t
        h_t = o_t * torch.tanh(c_t)
        return (h_t, c_t)


class GRUCell(nn.Module):
    """One time step of a GRU with the Module 2.1 equations (layer names as in Assignment 2)."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x2z = nn.Linear(input_size, hidden_size)
        self.h2z = nn.Linear(hidden_size, hidden_size, bias=False)
        self.x2r = nn.Linear(input_size, hidden_size)
        self.h2r = nn.Linear(hidden_size, hidden_size, bias=False)
        self.x2n = nn.Linear(input_size, hidden_size)
        self.h2n = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x_t, h_prev):
        z_t = torch.sigmoid(self.x2z(x_t) + self.h2z(h_prev))          # update gate
        r_t = torch.sigmoid(self.x2r(x_t) + self.h2r(h_prev))          # reset gate
        n_t = torch.tanh(self.x2n(x_t) + self.h2n(r_t * h_prev))       # candidate h~
        h_t = (1 - z_t) * h_prev + z_t * n_t
        return h_t


class RNNLM(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers, cell_type="lstm", dropout=0.1):
        super().__init__()
        self.is_lstm = (cell_type == "lstm")
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.tok_emb = nn.Embedding(vocab_size, hidden_size)
        cell_cls = LSTMCell if self.is_lstm else GRUCell
        self.cells = nn.ModuleList([cell_cls(hidden_size, hidden_size) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.tok_emb.weight)     # Module 2.1: Xavier/Glorot initialisation

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None, use_cache=False):
        """past_key_values here = list of per-layer states (h, c) / h, carried during generation.
        attention_mask (B, T): at padding positions the state is NOT updated."""
        B, T = input_ids.shape
        x = self.dropout(self.tok_emb(input_ids))
        new_states = []
        for l in range(self.num_layers):
            cell = self.cells[l]
            if past_key_values is not None:
                state = past_key_values[l]
            else:
                h = torch.zeros(B, self.hidden_size, device=x.device, dtype=x.dtype)
                state = (h, torch.zeros_like(h)) if self.is_lstm else h
            outputs = []
            for t in range(T):           # explicit unroll over time (BPTT through these steps)
                new_state = cell(x[:, t, :], state)
                if attention_mask is not None:
                    m = attention_mask[:, t].unsqueeze(1).to(x.dtype)     # 1 = real token, 0 = pad
                    if self.is_lstm:
                        new_state = (m * new_state[0] + (1 - m) * state[0], m * new_state[1] + (1 - m) * state[1])
                    else:
                        new_state = m * new_state + (1 - m) * state
                state = new_state
                outputs.append(state[0] if self.is_lstm else state)
            new_states.append(state)
            x = self.dropout(torch.stack(outputs, dim=1))    # input of the next layer

        if use_cache:
            x = x[:, -1:, :]
        logits = x @ self.tok_emb.weight.t()       # weight tying (Assignment 3)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)).float(),
                                   labels[:, 1:].reshape(-1))
        if use_cache:
            return {"loss": loss, "logits": logits, "past_key_values": new_states}
        return {"loss": loss, "logits": logits}

In [ ]:
class T5RelativePositionBias(nn.Module):
    def __init__(self, n_heads, bidirectional, num_buckets=32, max_distance=128):
        super().__init__()
        self.bidirectional = bidirectional
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.rel_emb = nn.Embedding(num_buckets, n_heads)     # one scalar per (bucket, head)

    def bucket(self, relative_position):
        # relative_position = key_pos - query_pos
        n = -relative_position
        num_buckets = self.num_buckets
        ret = torch.zeros_like(n)
        if self.bidirectional:                 # encoder: half the buckets for keys on the right
            num_buckets = num_buckets // 2
            ret = ret + (n < 0).long() * num_buckets
            n = torch.abs(n)
        else:                                  # decoder: only keys on the left exist
            n = torch.clamp(n, min=0)
        max_exact = num_buckets // 2           # small offsets get their own bucket
        is_small = n < max_exact
        # bigger offsets: logarithmically wider buckets, everything beyond max_distance shares the last one
        large = max_exact + (torch.log(torch.clamp(n, min=1).float() / max_exact)
                             / math.log(self.max_distance / max_exact)
                             * (num_buckets - max_exact)).long()
        large = torch.clamp(large, max=num_buckets - 1)
        return ret + torch.where(is_small, n, large)

    def forward(self, q_len, k_len, device):
        q_pos = torch.arange(q_len, device=device)[:, None]
        k_pos = torch.arange(k_len, device=device)[None, :]
        buckets = self.bucket(k_pos - q_pos)               # (q, k)
        bias = self.rel_emb(buckets)                       # (q, k, H)
        return bias.permute(2, 0, 1).unsqueeze(0)          # (1, H, q, k)


class T5Attention(nn.Module):
    """Self-attention (x_kv = x_q) or cross-attention (x_kv = encoder output)."""
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.attention_scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x_q, x_kv, position_bias=None, allowed=None):
        B, Tq, C = x_q.shape
        Tk = x_kv.size(1)
        q = self.q_proj(x_q).view(B, Tq, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) * self.attention_scale
        if position_bias is not None:
            scores = scores + position_bias
        if allowed is not None:
            scores = scores.masked_fill(~allowed, float("-inf"))
        att = self.attn_dropout(F.softmax(scores, dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, Tq, C)
        return self.out_proj(out)


class T5FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))   # dropout inside the FFN


class T5EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = T5Attention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = T5FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, position_bias, src_allowed):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, position_bias, src_allowed))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


class T5DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = T5Attention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.cross_attn = T5Attention(d_model, n_heads, dropout)
        self.norm3 = RMSNorm(d_model)
        self.ffn = T5FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, position_bias, causal_allowed, src_allowed):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, position_bias, causal_allowed))   # masked self-attention
        h = self.norm2(x)
        x = x + self.dropout(self.cross_attn(h, enc_out, None, src_allowed))        # cross-attention
        x = x + self.dropout(self.ffn(self.norm3(x)))
        return x


class T5Model(nn.Module):
    def __init__(self, vocab_size, pad_id, d_model, n_enc_layers, n_dec_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.pad_id = pad_id
        self.shared_emb = nn.Embedding(vocab_size, d_model)    # encoder input, decoder input and output
        self.dropout = nn.Dropout(dropout)
        self.enc_layers = nn.ModuleList([T5EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_enc_layers)])
        self.dec_layers = nn.ModuleList([T5DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_dec_layers)])
        self.enc_norm = RMSNorm(d_model)
        self.dec_norm = RMSNorm(d_model)
        self.enc_pos_bias = T5RelativePositionBias(n_heads, bidirectional=True)
        self.dec_pos_bias = T5RelativePositionBias(n_heads, bidirectional=False)
        nn.init.xavier_uniform_(self.shared_emb.weight)  # Module 2.1: Xavier/Glorot initialisation

    def encode(self, input_ids):
        S = input_ids.size(1)
        src_allowed = (input_ids != self.pad_id)[:, None, None, :]          # no attention to padding
        position_bias = self.enc_pos_bias(S, S, input_ids.device)
        x = self.dropout(self.shared_emb(input_ids))
        for layer in self.enc_layers:
            x = layer(x, position_bias, src_allowed)
        return self.dropout(self.enc_norm(x))

    def decode(self, decoder_input_ids, enc_out, input_ids, last_only=False):
        T = decoder_input_ids.size(1)
        src_allowed = (input_ids != self.pad_id)[:, None, None, :]
        causal_allowed = torch.tril(torch.ones(T, T, dtype=torch.bool, device=enc_out.device))[None, None]
        position_bias = self.dec_pos_bias(T, T, enc_out.device)
        x = self.dropout(self.shared_emb(decoder_input_ids))
        for layer in self.dec_layers:
            x = layer(x, enc_out, position_bias, causal_allowed, src_allowed)
        x = self.dropout(self.dec_norm(x))
        if last_only:
            x = x[:, -1:, :]
        return x @ self.shared_emb.weight.t()                                # weight sharing

    def forward(self, input_ids, decoder_input_ids, labels=None):
        enc_out = self.encode(input_ids)
        logits = self.decode(decoder_input_ids, enc_out, input_ids)
        loss = None
        if labels is not None:
            # decoder_input_ids are already shifted (<s> + target[:-1]); -100 = padding, ignored
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1), ignore_index=-100)
        return {"loss": loss, "logits": logits}

### 9.5 Load a Phase-1 model, add the NER head and the adaptation method

* **Hidden states:** we run the Phase-1 model without its LM output layer and take the final hidden state of every token. For `t5` we use only the encoder, which is bidirectional; its decoder is thrown away.
* **NER head:** one dropout and one linear layer, `hidden -> number of BIO tags`, trained with cross entropy. `O_WEIGHT` is the class weight of the `O` tag. With `HEAD_LOOKAHEAD = K` the linear layer reads `[h_t ; h_t+1 ; ... ; h_t+K]`. A left-to-right model has not seen the next words when it is at token t; the hidden states of the next K tokens have, so the head can tell where an entity starts and ends.
* **LoRA** [3]: `h = W x + (alpha/r) B A x`, where `A` (r x in) is random, `B` (out x r) starts at zero, and `W` is frozen.
* **LoRMA** [12] (multiplicative, additive-identity version): `h = (I + (alpha/r) B A) W x`, where `A` (r x out) is random, `B` (out x r) starts at zero, and `W` is frozen. The low-rank matrix multiplies the frozen weight instead of being added to it; the identity keeps the full rank, and `B = 0` means training starts exactly at the Phase-1 model.
* **partial:** freeze everything except the top `PARTIAL_TOP_LAYERS` layers, the final norm and the head. **head_only:** a linear probe of the frozen Phase-1 representations.
* **KD** [2]: the student is the first `STUDENT_LAYERS` layers of a Phase-1 model, initialised from its Phase-1 weights (like DistilBERT, Module 2.3). `loss = alpha * CE(gold) + (1 - alpha) * T^2 * KL(softmax(teacher/T) || softmax(student/T))`, with the teacher frozen. Every Phase-1 model uses the same BPE, so teacher and student tag the same tokens.

In [ ]:
import re
from safetensors.torch import load_file

def build_base(model_type, n_layers=None):
  V = len(bpe.itos)
  if model_type == "gpt":
    return DecoderLM(LMConfig(vocab_size=V, d_model=N_EMBD, n_layers=n_layers or N_LAYER, n_heads=N_HEAD,
                              n_kv_heads=N_HEAD, d_ff=D_FF, max_seq_len=MODEL_CONTEXT_LENGTH, dropout=DROPOUT,
                              pos_emb_type="learned", norm_type="layernorm", ffn_type="gelu"))
  if model_type == "modern":
    return DecoderLM(LMConfig(vocab_size=V, d_model=N_EMBD, n_layers=n_layers or N_LAYER, n_heads=N_HEAD,
                              n_kv_heads=N_KV_HEAD, d_ff=D_FF, max_seq_len=MODEL_CONTEXT_LENGTH, dropout=DROPOUT,
                              pos_emb_type="rope", norm_type="rmsnorm", ffn_type="swiglu"))
  if model_type == "rnn":
    return RNNLM(V, RNN_HIDDEN, n_layers or RNN_LAYERS, cell_type=RNN_CELL, dropout=DROPOUT)
  if model_type == "t5":
    return T5Model(V + T5_N_SENTINELS, bpe.pad_id, T5_D_MODEL, n_layers or T5_ENC_LAYERS, T5_DEC_LAYERS,
                   T5_HEADS, T5_D_FF, dropout=DROPOUT)
  raise ValueError(model_type)

def hidden_size(model_type):
  return {"gpt": N_EMBD, "modern": N_EMBD, "rnn": RNN_HIDDEN, "t5": T5_D_MODEL}[model_type]

def load_phase1_weights(base, model_type, n_layers=None):
  path = PHASE1_MODEL_FILES[model_type]
  state = load_file(path)
  if n_layers is not None:          # student: keep only the first n_layers layers
    kept = {}
    for k, v in state.items():
      m = re.match(r"^(blocks|cells|enc_layers)\.(\d+)\.", k)
      if m and int(m.group(2)) >= n_layers:
        continue
      kept[k] = v
    state = kept
  missing, unexpected = base.load_state_dict(state, strict=False)
  print(f"loaded Phase-1 {model_type} weights from {path} | missing {len(missing)} | unexpected {len(unexpected)}")
  if missing:
    raise ValueError(f"Phase-1 weights do not match the architecture, missing: {missing[:5]}")

def lm_hidden(base, input_ids):
  # DecoderLM.forward without the LM output layer
  B, T = input_ids.shape
  pos = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)
  x = base.tok_emb(input_ids)
  if base.cfg.pos_emb_type == "learned":
    x = x + base.pos_emb(pos)
  x = base.drop(x)
  for block in base.blocks:
    x, _ = block(x, pos)
  return base.ln_f(x)

def rnn_hidden(base, input_ids):
  # RNNLM.forward without the LM output layer
  B, T = input_ids.shape
  x = base.dropout(base.tok_emb(input_ids))
  for l in range(base.num_layers):
    cell = base.cells[l]
    h = torch.zeros(B, base.hidden_size, device=x.device, dtype=x.dtype)
    state = (h, torch.zeros_like(h)) if base.is_lstm else h
    outputs = []
    for t in range(T):
      state = cell(x[:, t, :], state)
      outputs.append(state[0] if base.is_lstm else state)
    x = base.dropout(torch.stack(outputs, dim=1))
  return x


class LoRALinear(nn.Module):
  """LoRA: h = W x + b + (alpha / r) * B A x  (W, b frozen)"""
  def __init__(self, base_linear, r, alpha):
    super().__init__()
    self.base = base_linear
    self.scale = alpha / r
    self.A = nn.Parameter(torch.randn(r, base_linear.in_features) / math.sqrt(base_linear.in_features))
    self.B = nn.Parameter(torch.zeros(base_linear.out_features, r))

  def forward(self, x):
    return self.base(x) + self.scale * ((x @ self.A.t()) @ self.B.t())


class LoRMALinear(nn.Module):
  """LoRMA (additive identity): h = (I + (alpha / r) * B A) W x + b  (W, b frozen)"""
  def __init__(self, base_linear, r, alpha):
    super().__init__()
    self.base = base_linear
    self.scale = alpha / r
    self.A = nn.Parameter(torch.randn(r, base_linear.out_features) / math.sqrt(base_linear.out_features))
    self.B = nn.Parameter(torch.zeros(base_linear.out_features, r))

  def forward(self, x):
    y = F.linear(x, self.base.weight)                      # W x
    y = y + self.scale * ((y @ self.A.t()) @ self.B.t())   # (I + s B A) W x
    if self.base.bias is not None:
      y = y + self.base.bias
    return y


def add_adapters(module, adapter_cls, r, alpha):
  # replace every nn.Linear inside the Phase-1 model by an adapter around it
  for name, child in module.named_children():
    if isinstance(child, nn.Linear):
      setattr(module, name, adapter_cls(child, r, alpha))
    else:
      add_adapters(child, adapter_cls, r, alpha)


class NERModel(nn.Module):
  def __init__(self, base, model_type, d_model, n_tags, dropout, class_weights, lookahead=0):
    super().__init__()
    self.base = base
    self.model_type = model_type
    self.lookahead = lookahead
    self.dropout = nn.Dropout(dropout)
    self.head = nn.Linear(d_model * (lookahead + 1), n_tags)
    self.register_buffer("class_weights", class_weights, persistent=False)

  def hidden(self, input_ids):
    if self.model_type == "t5":
      return self.base.encode(input_ids)                  # bidirectional encoder, padding masked
    if self.model_type == "rnn":
      return rnn_hidden(self.base, input_ids)
    return lm_hidden(self.base, input_ids)                # causal: right padding never affects real tokens

  def forward(self, input_ids, labels=None):
    h = self.hidden(input_ids)
    if self.lookahead > 0:
      B, T, D = h.shape
      parts = [h]
      for k in range(1, self.lookahead + 1):
        parts.append(torch.cat([h[:, k:, :], h.new_zeros(B, min(k, T), D)], dim=1)[:, :T, :])   # h_{t+k}, zeros past the end
      h = torch.cat(parts, dim=-1)
    logits = self.head(self.dropout(h))
    loss = None
    if labels is not None:
      loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1),
                             weight=self.class_weights, ignore_index=-100)
    return {"loss": loss, "logits": logits}


def set_trainable(ner, ft_method):
  for p in ner.parameters():
    p.requires_grad = (ft_method == "full")
  for p in ner.head.parameters():
    p.requires_grad = True
  if ft_method in ("lora", "lorma"):
    for name, p in ner.base.named_parameters():
      if name.endswith(".A") or name.endswith(".B"):
        p.requires_grad = True
  if ft_method == "partial":
    if ner.model_type in ("gpt", "modern"):
      layers, final_norm = ner.base.blocks, ner.base.ln_f
    elif ner.model_type == "rnn":
      layers, final_norm = ner.base.cells, None
    else:
      layers, final_norm = ner.base.enc_layers, ner.base.enc_norm
    for layer in list(layers)[-PARTIAL_TOP_LAYERS:]:
      for p in layer.parameters():
        p.requires_grad = True
    if final_norm is not None:
      for p in final_norm.parameters():
        p.requires_grad = True


def build_ner_model(model_type, ft_method, n_layers=None, load_pretrained=True):
  base = build_base(model_type, n_layers)
  if load_pretrained:
    load_phase1_weights(base, model_type, n_layers)
  if model_type == "t5":
    base.dec_layers = nn.ModuleList()       # NER uses only the T5 encoder
  if ft_method == "lora":
    add_adapters(base, LoRALinear, ADAPTER_RANK, ADAPTER_ALPHA)
  elif ft_method == "lorma":
    add_adapters(base, LoRMALinear, ADAPTER_RANK, ADAPTER_ALPHA)
  weights = torch.ones(len(TAGS))
  weights[TAG2ID["O"]] = O_WEIGHT
  ner = NERModel(base, model_type, hidden_size(model_type), len(TAGS), DROPOUT, weights, HEAD_LOOKAHEAD)
  set_trainable(ner, ft_method)
  return ner


class KDModel(nn.Module):
  def __init__(self, student, teacher, temperature, alpha):
    super().__init__()
    self.student = student
    self.teacher = teacher
    self.T = temperature
    self.alpha = alpha
    for p in self.teacher.parameters():
      p.requires_grad = False

  def train(self, mode=True):
    super().train(mode)
    self.teacher.eval()                     # the teacher never uses dropout
    return self

  def forward(self, input_ids, labels=None):
    out = self.student(input_ids, labels)
    if labels is None:
      return out
    with torch.no_grad():
      t_logits = self.teacher(input_ids)["logits"]
    mask = labels != -100
    s = out["logits"][mask].float()
    t = t_logits[mask].float()
    kd = F.kl_div(F.log_softmax(s / self.T, dim=-1), F.softmax(t / self.T, dim=-1),
                  reduction="batchmean") * self.T * self.T
    return {"loss": self.alpha * out["loss"] + (1 - self.alpha) * kd, "logits": out["logits"]}


if USE_KD:
  student = build_ner_model(MODEL_TYPE, FT_METHOD, n_layers=STUDENT_LAYERS)
  teacher = build_ner_model(TEACHER_MODEL_TYPE, TEACHER_FT_METHOD, load_pretrained=False)
  teacher.load_state_dict(torch.load(TEACHER_NER_PATH, map_location="cpu"))
  print("teacher loaded from", TEACHER_NER_PATH)
  model = KDModel(student, teacher, KD_TEMPERATURE, KD_ALPHA)
  ner_model = student
  n_teacher = sum(p.numel() for p in teacher.parameters())
else:
  ner_model = build_ner_model(MODEL_TYPE, FT_METHOD)
  model = ner_model
  n_teacher = 0

n_total = sum(p.numel() for p in ner_model.parameters())
n_trainable = sum(p.numel() for p in ner_model.parameters() if p.requires_grad)
print(f"NER model: {n_total / 1e6:.2f}M parameters | trainable {n_trainable / 1e6:.3f}M ({100 * n_trainable / n_total:.2f}%)")
if USE_KD:
  print(f"teacher: {n_teacher / 1e6:.2f}M parameters -> student is {100 * n_total / n_teacher:.0f}% of the teacher")

## 10. Training (Trainer, like Phase 1)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
  output_dir=os.path.join(EXP_DIR, "checkpoints"),
  num_train_epochs=EPOCHS,
  per_device_train_batch_size=BATCH_SIZE,
  per_device_eval_batch_size=BATCH_SIZE,
  learning_rate=LEARNING_RATE,
  weight_decay=0.01,
  warmup_steps=5 if SMOKE_TEST else 50,
  lr_scheduler_type="cosine",
  max_grad_norm=1.0,
  fp16=torch.cuda.is_available() and MODEL_TYPE != "t5",   # T5 trains in full precision (fp16 broke it)
  eval_strategy="epoch",
  save_strategy="epoch",
  save_total_limit=2,
  load_best_model_at_end=True,          # checkpoint selection on validation loss
  metric_for_best_model="eval_loss",
  greater_is_better=False,
  logging_steps=20,
  seed=SEED,
  report_to="none",
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset,
                  eval_dataset=val_dataset, data_collator=data_collator)

if torch.cuda.is_available():
  torch.cuda.reset_peak_memory_stats()
start_time = time.time()
if LOAD_NER_PATH:
  ner_model.load_state_dict(torch.load(LOAD_NER_PATH, map_location="cpu"))
  print("loaded trained NER model from", LOAD_NER_PATH, "- skipping training")
else:
  trainer.train()
training_time = time.time() - start_time
peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f"Training time: {training_time / 60:.1f} min | peak GPU memory: {peak_memory_gb:.2f} GB")
for h in trainer.state.log_history:
  if "eval_loss" in h:
    print(f"epoch {h['epoch']:.0f}: eval_loss {h['eval_loss']:.4f}")
  elif "loss" in h:
    print(f"   step {h['step']}: train loss {h['loss']:.4f}")

## 11. Validation: strict macro-F1 and error analysis

Prediction runs on the whole validation documents, exactly like the test set. The strict score counts a span as correct only when start, end and label all match. We also sort every mistake into a type: boundary error (right label, wrong offsets), label confusion (right offsets, wrong label), boundary and label both wrong, spurious span, or missed entity.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ner_model = ner_model.to(device)
ner_model.eval()
PRED_BATCH = 16

@torch.no_grad()
def predict_document(doc):
  ids, starts, ends = doc_tok[doc]
  n = len(ids)
  best_score = [-2] * n
  best_tag = [0] * n
  wins = window_starts(n)
  for b in range(0, len(wins), PRED_BATCH):
    chunk = wins[b:b + PRED_BATCH]
    x = torch.full((len(chunk), WINDOW_TOKENS), bpe.pad_id, dtype=torch.long)
    for j, st in enumerate(chunk):
      seg = ids[st:st + WINDOW_TOKENS]
      x[j, :len(seg)] = torch.tensor(seg, dtype=torch.long)
    if device.type == "cuda" and MODEL_TYPE != "t5":
      with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = ner_model(x.to(device))["logits"]
    else:
      logits = ner_model(x.to(device))["logits"]
    pred = logits.argmax(dim=-1).tolist()
    for j, st in enumerate(chunk):
      L = min(WINDOW_TOKENS, n - st)
      for k in range(L):
        score = k if CAUSAL else min(k, L - 1 - k)      # most context on the left / on both sides
        if CAUSAL and HEAD_LOOKAHEAD > 0 and (L - 1 - k) < HEAD_LOOKAHEAD and st + L < n:
          score = -1                                    # the lookahead would run past this window
        if score > best_score[st + k]:
          best_score[st + k] = score
          best_tag[st + k] = pred[j][k]
  text = texts[doc] if doc in texts else test_texts[doc]
  return snap_spans(text, decode_spans(best_tag, starts, ends))

t0 = time.time()
val_pred = {doc: predict_document(doc) for doc in val_docs}
val_seconds = time.time() - t0
val_gold = {doc: gold[doc] for doc in val_docs}
mp, mr, mf, per_label = strict_scores(val_gold, val_pred)
print(f"VALIDATION strict macro: P {mp:.4f} | R {mr:.4f} | F1 {mf:.4f}   ({val_seconds:.1f}s for {len(val_docs)} docs)")
print(f"{'label':10s} {'P':>6s} {'R':>6s} {'F1':>6s} {'#gold':>6s}")
for l, (p, r, f, n) in sorted(per_label.items(), key=lambda kv: kv[1][2]):
  print(f"{l:10s} {p:6.3f} {r:6.3f} {f:6.3f} {n:6d}")

# ---- error analysis ----
err = {"boundary (right label)": 0, "label confusion (right span)": 0, "boundary + label": 0,
       "spurious (no gold overlap)": 0, "missed gold entity": 0}
confusions, boundary_examples, spurious_texts = {}, [], {}
for doc in val_docs:
  t = texts[doc]
  g = set(val_gold[doc])
  p = set(val_pred[doc])
  for (s, e, lab) in p - g:
    overlaps = [x for x in g if x[0] < e and s < x[1]]
    if any(x[0] == s and x[1] == e for x in overlaps):
      gl = [x for x in overlaps if x[0] == s and x[1] == e][0][2]
      err["label confusion (right span)"] += 1
      confusions[(gl, lab)] = confusions.get((gl, lab), 0) + 1
    elif any(x[2] == lab for x in overlaps):
      err["boundary (right label)"] += 1
      x = [x for x in overlaps if x[2] == lab][0]
      if len(boundary_examples) < 10:
        boundary_examples.append((lab, t[s:e], t[x[0]:x[1]]))
    elif overlaps:
      err["boundary + label"] += 1
    else:
      err["spurious (no gold overlap)"] += 1
      key = (lab, t[s:e][:40])
      spurious_texts[key] = spurious_texts.get(key, 0) + 1
  for (s, e, lab) in g - p:
    if not any(x[0] < e and s < x[1] for x in p):
      err["missed gold entity"] += 1
print("\nerror types:", err)
print("most confused labels (gold -> predicted):", sorted(confusions.items(), key=lambda kv: -kv[1])[:8])
print("boundary errors (label, predicted text, gold text):")
for b in boundary_examples:
  print("  ", b)
print("most frequent spurious predictions (boilerplate?):", sorted(spurious_texts.items(), key=lambda kv: -kv[1])[:8])

## 12. Save the NER model (needed when this experiment is the KD teacher)

In [ ]:
torch.save(ner_model.state_dict(), os.path.join(EXP_DIR, "ner_model.pt"))
print("saved", os.path.join(EXP_DIR, "ner_model.pt"))

## 13. Test predictions -> prediction.json -> submission zip

In [ ]:
t0 = time.time()
predictions = {}
for doc in test_texts:
  spans = predict_document(doc)
  out = []
  for s, e, lab in spans:
    out.append([s, e - 1 if END_INCLUSIVE else e, lab, test_texts[doc][s:e]])
  predictions[doc] = out
test_seconds = time.time() - t0

# format checks before writing
assert set(predictions.keys()) == set(test_texts.keys())
for doc, spans in predictions.items():
  for s, e, lab, txt in spans:
    e_excl = e + 1 if END_INCLUSIVE else e
    assert 0 <= s < e_excl <= len(test_texts[doc]) and lab in LABELS
    assert test_texts[doc][s:e_excl] == txt and txt == txt.strip()

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
  json.dump(predictions, f, ensure_ascii=False)
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
  z.write(PREDICTION_PATH, arcname="prediction.json")
n_spans = sum(len(v) for v in predictions.values())
print(f"{n_spans} entities predicted for {len(predictions)} test docs in {test_seconds:.0f}s")
print("submission:", SUBMISSION_ZIP)

## 14. Log this experiment (one line per experiment in results.jsonl)

In [ ]:
result = {
  "experiment": EXPERIMENT_NAME, "model": MODEL_TYPE, "method": FT_METHOD, "kd": USE_KD,
  "student_layers": STUDENT_LAYERS if USE_KD else None,
  "teacher": f"{TEACHER_MODEL_TYPE}/{TEACHER_FT_METHOD}" if USE_KD else None,
  "params_M": round(n_total / 1e6, 2), "trainable_M": round(n_trainable / 1e6, 3),
  "trainable_pct": round(100 * n_trainable / n_total, 2),
  "train_min": round(training_time / 60, 1), "peak_mem_GB": round(peak_memory_gb, 2),
  "test_pred_s": round(test_seconds, 1),
  "val_macro_P": round(mp, 4), "val_macro_R": round(mr, 4), "val_macro_F1": round(mf, 4),
  "per_label_F1": {l: round(v[2], 3) for l, v in per_label.items()},
  "errors": err, "lookahead": HEAD_LOOKAHEAD, "window": WINDOW_TOKENS, "split_mode": SPLIT_MODE, "o_weight": O_WEIGHT,
  "lr": LEARNING_RATE, "epochs": EPOCHS, "rank": ADAPTER_RANK if FT_METHOD in ("lora", "lorma") else None,
  "offset_ceiling_F1": round(rf, 4),
  "decode_merge": DECODE_MERGE, "snap": SNAP_TO_WORDS,
}
with open(RESULTS_PATH, "a") as f:
  f.write(json.dumps(result) + "\n")
print(json.dumps(result, indent=1))